## ASQCI for the star Hamiltonian family

In [ ]:
import numpy as np
import matplotlib.pyplot as plt, scienceplots
plt.style.use(["science"])

In [ ]:
def star_hamiltonian(leaves):
    """Single-excitation sector of Eq. (34): index 0 is the hub, indices 1..leaves are the leaves."""
    hamiltonian = np.diag(np.r_[0.0, 8 * leaves, np.ones(leaves - 1)])
    hamiltonian[0, 1:] = hamiltonian[1:, 0] = -np.r_[1.0, 1 - np.arange(2, leaves + 1) / (2 * leaves)]
    return hamiltonian


def adapt_distributions(hamiltonian):
    """distributions[j - 1, z] = |<z|psi_j>|^2, with psi_j the ground state on the hub and the first j leaves."""
    size = len(hamiltonian)
    distributions = np.zeros((size - 1, size))
    for depth in range(1, size):
        distributions[depth - 1, : depth + 1] = np.linalg.eigh(hamiltonian[: depth + 1, : depth + 1])[1][:, 0] ** 2
    return distributions


def expected_error(hamiltonian, distributions, shots):
    """Exact E[E(S) - E0], where S is config 0 plus every config seen in shots[..., j] samples of distributions[j]."""
    others = len(hamiltonian) - 1
    subsets = np.arange(2**others)
    member = (subsets[:, None] >> np.arange(others)) & 1  # member[K, i] = 1 if config i + 1 is in subset K
    coefficient = np.array([np.linalg.eigvalsh(hamiltonian[np.ix_(keep, keep)])[0] for keep in (np.r_[0, 1 + np.flatnonzero(row)] for row in member)])
    coefficient -= np.linalg.eigvalsh(hamiltonian)[0]
    for bit in range(others):  # Moebius transform, so that E[error] = sum_K coefficient[K] * Pr[all seen configs lie in K]
        without = subsets[(subsets >> bit) & 1 == 0]
        coefficient[without] -= coefficient[without | (1 << bit)]
    log_probability = np.log1p(-distributions[:, 1:] @ (1 - member.T))  # one shot of circuit j stays inside subset K
    return np.exp(shots @ log_probability) @ coefficient

In [ ]:
shots_per_iteration = np.unique(np.logspace(0, 4, 200).astype(int))
fig, (ax_error, ax_ratio) = plt.subplots(2, 1, sharex=True, figsize=(4, 4))
for leaves, color in zip((2, 4, 8), ("C0", "C1", "C2")):
    hamiltonian = star_hamiltonian(leaves)
    distributions = adapt_distributions(hamiltonian)
    pooled = expected_error(hamiltonian, distributions, shots_per_iteration[:, None] * np.ones(leaves))
    single = np.min([expected_error(hamiltonian, distributions, leaves * shots_per_iteration[:, None] * row) for row in np.eye(leaves)], axis=0)
    ax_error.plot(shots_per_iteration, pooled, color=color, label=f"ASQCI, $p={leaves}$")
    ax_error.plot(shots_per_iteration, single, color=color, ls="--", label=f"QSCI, $p={leaves}$")
    ax_ratio.plot(shots_per_iteration, pooled / single, color=color, label=f"$p={leaves}$")
    guaranteed = int(np.ceil(2560 * leaves**2 * np.log(8192 * leaves**5)))
ax_error.set(xscale="log", yscale="log", ylim=(1e-8, 10), ylabel=r"$\mathbb{E}[E_p - E_0]$")
ax_ratio.axhline(1, color="k", ls=":", lw=0.8)
ax_ratio.set(xscale="log", yscale="log", ylim=(1e-2, 10), xlabel="Shots per iteration $N$", ylabel="ASQCI error / QSCI error")
ax_error.legend(fontsize=6); ax_ratio.legend(fontsize=6)
fig.tight_layout()
# fig.savefig("star-result.pdf")